In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-02-017/MGC+01-02-017_1/FORS2.2021-10-04T04_41_52.988/MCG+01-02-017_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-02-017/MGC+01-02-017_1/FORS2.2021-10-04T04_41_52.988/MCG+01-02-017_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     361   (2094, 963)   float32   
  1  IMAGE.ERR     1 ImageHDU        42   (2094, 963)   float32   


In [3]:
image_cut = image[61:161, 0:1890]
image_error_cut = image_error[61:161, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.0429
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) #+
        #ersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-02-017/PSF/Real_seeing_MCG+01-02-017.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    #purple_area = single_integral(x, params6, a, b)
    #pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area# + purple_area #+ pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area]#, (purple_area*100)/sum_area]#, (pink_area*100)/sum_area]

## OIII

In [8]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit_OIII = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-02-017/MGC+01-02-017_1/FORS2.2021-10-04T04_41_52.988/Fit/OIII_fit.csv", index_col = 0)

In [9]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -sigma, fit_OIII['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -sigma, fit_OIII['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -sigma, fit_OIII['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -sigma, fit_OIII['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -sigma, fit_OIII['Component 5'].iloc[3] + sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -sigma, fit_OIII['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

0.18532887771256787
0.18679435122727694
0.1766508967478974
0.17666587232948755
0.45196822629156896
0.4527520280672075
1.3864181313626047
1.3963471104969702
1.1806228925451163
0.7490423113466704
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       63.015444   5.113293   1.942767   0.024845  4.701948e-04
Green       3.244179  27.265978  17.056101   0.016730  8.623674e-11
Magenta     0.000000   0.691172  15.854838   0.000000  0.000000e+00
Orange     32.847485  65.419691  63.761662  98.786886  1.555813e+01
Chocolate   0.892892   1.509866   1.384631   1.171539  8.444140e+01


In [10]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -2*sigma, fit_OIII['Component 5'].iloc[3] + 2*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -2*sigma, fit_OIII['Component 6'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

0.8788619294069846
0.8834869024917432
0.8926328245035579
0.8935198239822002
1.1095666656726686
1.1106308551672828
2.6765780707944815
2.6923084574132496
2.07307232121127
1.3889167582270512
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       60.779391   5.193796   1.870527   0.025997  5.400197e-04
Green       3.583664  26.424266  16.438928   0.018764  1.317177e-10
Magenta     0.000000   1.976730  13.421756   0.000000  0.000000e+00
Orange     34.694518  64.909010  66.830418  98.739176  1.774301e+01
Chocolate   0.942426   1.496198   1.438371   1.216064  8.225645e+01


In [11]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -3*sigma, fit_OIII['Component 5'].iloc[3] + 3*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -3*sigma, fit_OIII['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

1.1780825848616325
1.182660256426104
1.2634582046952005
1.2653533783891915
1.5179912074870316
1.5196056001859581
4.3611515689389835
4.378122139553589
3.1157816783194052
2.3237187404398094
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       58.782839   5.285561   1.963714   0.030775  7.426810e-04
Green       3.926462  25.574738  16.695538   0.030078  4.450830e-10
Magenta     0.000000   3.264545  11.225293   0.000000  0.000000e+00
Orange     36.305287  64.392894  68.640846  98.660094  2.372917e+01
Chocolate   0.985412   1.482262   1.474609   1.279054  7.627009e+01


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-02-017/MGC+01-02-017_1/FORS2.2021-10-04T04_41_52.988/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 1'].iloc[3] -sigma, fit_OIII['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 2'].iloc[3] -sigma, fit_OIII['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 3'].iloc[3] -sigma, fit_OIII['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 4'].iloc[3] -sigma, fit_OIII['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 5'].iloc[3] -sigma, fit_OIII['Component 5'].iloc[3] + sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_OIII['Component 6'].iloc[3] -sigma, fit_OIII['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

0.19467371912530806
0.19478653563764353
0.1470819123964453
0.14701943959091818
0.338744021107831
0.38020294810792327
1.3088001693037181
1.3214347895183094
0.7757830341876896
0.7426663237403963
                 Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       7.445432e+01   1.357772   0.230589   0.000008  1.013404e-11
Green      1.652701e-08  30.312314  22.804273   0.000000  0.000000e+00
Magenta    2.568445e-04   0.028655   1.560430   0.000089  3.032163e-06
Orange     2.550975e+01  68.167294  75.241499  99.714910  1.999757e+01
Chocolate  3.568019e-02   0.133966   0.163209   0.284993  8.000243e+01


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_OIII['Component 5'].iloc[3] -2*sigma, fit_OIII['Component 5'].iloc[3] + 2*sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
#                     fit_OIII['Component 6'].iloc[3] -2*sigma, fit_OIII['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

0.9114193897926143
0.9115724875680877
0.737994214163538
0.738219083822729
0.8396691664572311
0.960839339585791
2.52742655589736
2.547450984857571
1.4283934688915747
1.375863095720712
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       72.660758   1.506458   0.216510   0.000009  1.228054e-11
Green       0.000362  30.166086  19.471984   0.000000  0.000000e+00
Magenta     0.000285   0.052140   1.832223   0.000094  3.309342e-06
Orange     27.300324  68.141075  78.307224  99.703023  2.175338e+01
Chocolate   0.038272   0.134241   0.172059   0.296874  7.824662e+01


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 5'].iloc[3] -3*sigma, fit_OIII['Component 5'].iloc[3] + 3*sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
#                     fit_OIII['Component 6'].iloc[3] -3*sigma, fit_OIII['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

1.2037524861554918
1.2036778752192956
1.0367833849818386
1.0400577508836106
1.1623319585661698
1.3004932427975524
4.1195682942899
4.1414817350806565
2.3354893134555614
2.2739753037738475
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       70.951197   1.677878   0.248136   0.000015  2.234170e-11
Green       0.003486  29.949091  18.474027   0.000000  0.000000e+00
Magenta     0.000314   0.101352   1.575771   0.000118  4.125854e-06
Orange     29.004240  68.137091  79.526892  99.691843  2.676334e+01
Chocolate   0.040763   0.134588   0.175175   0.308024  7.323665e+01
